# 15 · Gui.Desk 实时 —— 策略进 desk 的「实时图表」，按策略真下单

**模式：desk 界面 + 实时（快期模拟盘）**
运行：`python tutorials/plots/15_desk_live.py --user 账号 --password 密码`
（或设环境变量 `TQ_USER` / `TQ_PASS`）

* `Bt(live=True).run(gui=Gui.Desk)` —— desk 复用这个 `TqApi`，打开
  **实时图表** 并「加到图表」；图上按策略**真实下单**（默认快期模拟盘）；
* desk 底部账户面板能看到持仓 / 委托 / 成交；工具栏可切周期 / 指标 / 布局；
* 关窗（或 Ctrl+C）→ 停策略并关闭天勤连接。

> **Jupyter**：先在下面「账号」一格填账号（或设环境变量），运行后 desk 会以
> **iframe** 内嵌实时图表。交易时段外能连上但没有行情。
>
> ⚠ 这是**快期模拟盘**；实盘账户（`TqAccount`）写法见
> `tutorials/desk/gui/06_live_single_shfe.py`。


In [ ]:
import os
import sys

# 找到仓库根目录（notebook 的 cwd 通常就是 tutorials/plots）→ 优先用仓库里的
# minibt，避免被 site-packages 里的旧副本遮蔽
_here = os.path.abspath(os.getcwd())
while _here and not os.path.isdir(os.path.join(_here, 'minibt')):
    _parent = os.path.dirname(_here)
    if _parent == _here:
        break
    _here = _parent
if os.path.isdir(os.path.join(_here, 'minibt')):
    sys.path.insert(0, _here)

import numpy as np

from minibt import *                      # noqa: F401,F403
from minibt import (BtIndicator, Colors, Config, Gui, LineDash, LineStyle,
                    Markers, SignalStyle, Strategy)

#: ★ 上期所主力合约（白银）；换黄金：SHFE.au2612
SYMBOL = 'SHFE.ag2612'


In [ ]:
class LiveMa(BtIndicator):
    """双均线 + 交叉箭头（实时图上用的指标）。"""

    lines = ('fast', 'slow', 'long_signal', 'short_signal')
    params = dict(fast=10, slow=30, cooldown=30)
    overlap = True
    isplot = dict(long_signal=False, short_signal=False)

    linestyle = dict(
        fast=LineStyle(line_color='#f6c343', line_width=2),
        slow=LineStyle(line_color='#2196f3', line_width=2,
                       line_dash=LineDash.dashed),
    )
    signalstyle = dict(
        long_signal=SignalStyle('low', Colors.lime, Markers.arrow_up,
                                size=12).set_label('B', 11, color='#26a69a'),
        short_signal=SignalStyle('high', Colors.tomato, Markers.arrow_down,
                                 size=12).set_label('S', 11, color='#ef5350'),
    )

    def next(self):
        fast = self.close.rolling(int(self.params.fast)).mean().to_numpy()
        slow = self.close.rolling(int(self.params.slow)).mean().to_numpy()
        diff = fast - slow
        prev = np.concatenate(([np.nan], diff[:-1]))
        up = np.where((prev <= 0) & (diff > 0), 1.0, 0.0)
        dn = np.where((prev >= 0) & (diff < 0), -1.0, 0.0)
        return fast, slow, up, dn


class LiveMaStrategy(Strategy):
    """实时策略：金叉多 1 手、死叉平仓。

    ``params.symbol`` 一定要写具体合约：desk 靠它决定打开哪个实时图表页
    （主连 / 指数 ``KQ.*`` 不可交易，只适合看行情 ✓）。
    """

    config = Config(value=1_000_000.0, margin_rate=0.05,
                    fixed_commission=1.0)
    params = dict(symbol=SYMBOL, size=1)

    def __init__(self):
        self.min_start_length = 60
        self.data = self.get_kline(self.params.symbol)
        self.ma = LiveMa(self.data)

    def next(self):
        position = float(getattr(self.data.position, 'pos',
                                 self.data.position) or 0.0)
        if self.ma.long_signal.new and position <= 0:
            self.data.set_target_size(int(self.params.size))
        elif self.ma.short_signal.new and position > 0:
            self.data.set_target_size(0)


In [ ]:
# ★ 账号：推荐设环境变量 TQ_USER / TQ_PASS；
#   也可以直接把下面的 '' 改成你的天勤账号 / 密码（改完别提交到仓库）
USER = os.environ.get('TQ_USER', '')
PASSWORD = os.environ.get('TQ_PASS', '')

if not USER or not PASSWORD:
    print('跳过：请先填 USER / PASSWORD，或设置环境变量 TQ_USER / TQ_PASS')
else:
    from tqsdk import TqApi, TqAuth, TqKq
    # 快期模拟盘（不动真钱）；实盘账户写法见 tutorials/desk/gui/06
    api = TqApi(TqKq(), auth=TqAuth(USER, PASSWORD))

    strategy = LiveMaStrategy.copy(name='LiveMaStrategy',
                                   params=dict(symbol=SYMBOL, size=1))
    bt = Bt(auto=False, live=True)      # auto=False：只跑这一个策略
    bt.addTqapi(api=api)
    bt.addstrategy(strategy)
    bt.run(gui=Gui.Desk)                # 实时图表「加到图表」= 按策略下单


## 说明

* 默认**快期模拟盘**（`TqKq`），不动真钱；
* 图上按策略真下单：底部账户面板能看到持仓 / 委托 / 成交；
* 关闭 desk（iframe 里关不掉时：`Ctrl+C` 停 kernel 或重启 kernel）会一并
  关闭天勤连接；
* `params.symbol` 写具体月份合约；`KQ.m@…`（主连）/ `KQ.i@…`（指数）不可交易；
* 换引擎只改 `gui`：`Gui.Pylightcharts`（原生窗口）/
  `Gui.PylightchartsWeb`（浏览器标签）。
* `SYMBOL` 这种常量写在**上一格**也没问题：把策略交给 desk 时，会把
  notebook 里用到、但策略格没定义的字面量常量一起带到生成的 `.py` 里
  （只带 str / int / float / bool / None 及 tuple / list / dict，
  不会把上一格的代码再执行一遍 ✓）。
